In [ ]:
import os

SEED = 0 


os.environ["PYTHONHASHSEED"] = str(SEED)
os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8") 


# -------------------------------------------------------------
# now import libraries
import random
import numpy as np

random.seed(SEED)
np.random.seed(SEED)

# Torch
import torch
torch.manual_seed(SEED)


import scanpy as sc, anndata as ad, numpy as np, pandas as pd
import logging
import triku as tk 
from matplotlib import pylab
import os
import sys
import yaml
from scipy.sparse import csr_matrix
import gc
import torch
import scanit
from scipy.sparse import issparse
import scanit
from scipy.sparse import issparse
import gc
import torch

In [ ]:
nThreads = 10
import os

os.environ["OMP_NUM_THREADS"] = f"{nThreads}"
os.environ["OPENBLAS_NUM_THREADS"] = f"{nThreads}"
os.environ["MKL_NUM_THREADS"] = f"{nThreads}"
os.environ["BLIS_NUM_THREADS"] = f"{nThreads}"
os.environ["VECLIB_MAXIMUM_THREADS"] = f"{nThreads}"
os.environ["MKL_DYNAMIC"] = "FALSE"


In [ ]:

homeDir = os.getenv("HOME")

sys.path.insert(1, homeDir+"/utils/")


from PlotPCA_components import *
from AdataSanityCheck import *
from PurgeAdata import *
from spatialUtils import *
from _DEAplots import *
from _Aggregation import *
from _DEGs_utils import *
from _plotting import *
import rapids_singlecell as rsc
import ipynbname
import nbconvert.exporters
from nbconvert.preprocessors import TagRemovePreprocessor
import os


try:
    nb_name = ipynbname.name()
except:
    nb_name = "".join(os.path.basename(globals()['__vsc_ipynb_file__']))

print(nb_name)



In [ ]:
with open(homeDir+"/utils/config.yaml", 'r') as f:
    analysis_params = yaml.safe_load(f)["analysisParams"]
print(analysis_params)
DS = "B19-25653_8um_banksy"
DSname = "Banksy_B19-25653_8um"
FigTag =  "B19-25653"
base_path = "/data/Spatial_Tx" 
HashesDir = homeDir+"/hashes"
Celltype = "Tumorcells"

import cupyx.scipy.sparse
import random
from scipy import sparse
import rmm
from rmm.allocators.cupy import rmm_cupy_allocator
import cupy as cp

cp.cuda.set_allocator(rmm_cupy_allocator)

In [ ]:
%load_ext rpy2.ipython
pd.DataFrame.iteritems = pd.DataFrame.items

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from matplotlib import colormaps
from matplotlib.colors import Normalize, to_hex
from matplotlib.patches import Patch
from mpl_toolkits.axes_grid1 import make_axes_locatable
from pandas.api import types as ptypes


def plot_spatial_obs(
    adata,
    obs: str,
    library_id: str = "{DS}_hires_image",
    img_key: str = "hires",
    scale_key: str = "tissue_hires_scalef",
    spatial_key: str = "spatial",
    dotscale: float = 1.0,
    vmax_quantile: float = 0.99,
    vmin=None,
    vmax=None,
    marker: str = "s",
    img_alpha: float = 0.7,
    show_colorbar: bool = True,
    groups: list = None,
    linewidths: float = 0,
    width: int = 5,
    dpi: int = 100,
    ax=None,
    palette=None,
    show_legend: bool = True,
    legend_kwargs: dict = None,
    crop_image: dict = None,
    save: str = None,
):
    if legend_kwargs is None:
        legend_kwargs = {}

    # Extract image and scaling
    img = adata.uns[spatial_key][library_id]["images"][img_key]
    img_height, img_width = img.shape[:2]
    aspect = img_width / img_height

    scale = adata.uns[spatial_key][library_id]["scalefactors"][scale_key]
    diameter_px = adata.uns[spatial_key][library_id]["scalefactors"][
        "spot_diameter_fullres"
    ]

    # Extract spatial coordinates and obs values
    mask = adata.obs[obs].isin(groups) if groups is not None else adata.obs[obs].notna()
    coords = adata[mask].obsm[spatial_key] * scale
    obs_values = adata[mask].obs[obs]

    # Apply cropping if specified
    if crop_image is not None:
        xmin = int(crop_image["xmin"])
        xmax = int(crop_image["xmax"])
        ymin = int(crop_image["ymin"])
        ymax = int(crop_image["ymax"])

        img = img[ymin:ymax, xmin:xmax, ...]

        crop_mask = (
            (coords[:, 0] >= xmin)
            & (coords[:, 0] < xmax)
            & (coords[:, 1] >= ymin)
            & (coords[:, 1] < ymax)
        )

        coords = coords[crop_mask] - np.array([xmin, ymin])
        obs_values = obs_values[crop_mask]

        img_height, img_width = img.shape[:2]
        aspect = img_width / img_height

    # Figure / axis
    external_ax = ax is not None

    if not external_ax:
        fig, ax = plt.subplots(
            figsize=(width + 0.8, width / aspect),
            dpi=dpi,
        )
    else:
        fig = ax.figure

    ax.imshow(img, origin="upper", alpha=img_alpha)

    # Match plot_spatial_gene's spot sizing exactly.
    px_to_pt = 72 / dpi
    marker_size_pt2 = (diameter_px * px_to_pt / 2 * dotscale) ** 2

    if ptypes.is_numeric_dtype(obs_values):
        if vmin is None:
            vmin = 0
        if vmax is None:
            vmax = np.quantile(obs_values, vmax_quantile)

        norm = Normalize(vmin=vmin, vmax=vmax)
        cmap = colormaps.get_cmap(palette)

        sc = ax.scatter(
            coords[:, 0],
            coords[:, 1],
            c=obs_values,
            cmap=cmap,
            norm=norm,
            s=marker_size_pt2,
            linewidths=linewidths,
            edgecolors="black",
            marker=marker,
        )

        if show_colorbar:
            divider = make_axes_locatable(ax)
            cbar_ax = divider.append_axes("right", size="3%", pad=0.12)
            fig.colorbar(sc, cax=cbar_ax)
            cbar_ax.set_ylabel(
                str(obs),
                fontsize=12,
                rotation=270,
                labelpad=15,
            )
            cbar_ax.tick_params(labelsize=10)

    else:
        obs_cat = obs_values.astype("category").cat.remove_unused_categories()
        categories_present = list(obs_cat.cat.categories)

        if f"{obs}_colors" in adata.uns:
            all_categories = list(adata.obs[obs].astype("category").cat.categories)
            all_colors = [to_hex(c) for c in adata.uns[f"{obs}_colors"]]
            global_map = dict(zip(all_categories, all_colors))
            color_dict = {
                category: global_map.get(category, "#808080")
                for category in categories_present
            }
        else:
            cmap = colormaps.get_cmap(palette or "tab20")
            color_dict = {
                category: to_hex(cmap(i / max(len(categories_present) - 1, 1)))
                for i, category in enumerate(categories_present)
            }

        colors = obs_cat.map(color_dict).values

        ax.scatter(
            coords[:, 0],
            coords[:, 1],
            c=colors,
            s=marker_size_pt2,
            linewidths=linewidths,
            edgecolors="black",
            marker=marker,
        )

        if show_legend:
            handles = [
                Patch(
                    facecolor=color_dict[category],
                    edgecolor="black",
                    label=str(category),
                )
                for category in categories_present
            ]
            legend_options = {
                "loc": "center left",
                "bbox_to_anchor": (1.01, 0.5),
                "frameon": False,
            }
            legend_options.update(legend_kwargs)
            ax.legend(handles=handles, **legend_options)

    ax.axis("off")

    if not external_ax:
        plt.tight_layout()

    if save is not None:
        fig.savefig(save, dpi=dpi, bbox_inches="tight")

    if not external_ax:
        plt.show()


# Load data

In [ ]:
adataKnn = sc.read_h5ad(f"./{DS}_KNNmetaspots.h5ad")



In [ ]:
if "leiden_colors" in adataKnn.uns:
    del adataKnn.uns["leiden_colors"]
sc.pl.umap(adataKnn, color="leiden")

In [ ]:
plot_spatial_obs(
    adataKnn, obs="leiden", width=7, dpi=200,dotscale=4 ,img_key="hires",marker='o',linewidths=0,
    spatial_key='spatial',scale_key="tissue_hires_scalef",library_id= f"{FigTag}_hires_image",  img_alpha=.4,  legend_kwargs={"fontsize":10})
for leidenlabel  in adataKnn.obs["leiden"].unique():
    plot_spatial_obs(
        adataKnn, obs="leiden", width=7, dpi=200,dotscale=4 ,img_key="hires",marker='o',linewidths=0,groups=[leidenlabel],
        spatial_key='spatial',scale_key="tissue_hires_scalef",library_id= f"{FigTag}_hires_image", img_alpha=.4,  legend_kwargs={"fontsize":10})

# 1) Scanpy

In [ ]:
if "rank_genes_groups" in adataKnn.uns:
    del adataKnn.uns["rank_genes_groups"]
if "dendrogram_leiden" in adataKnn.uns:
    del adataKnn.uns["dendrogram_leiden"]


sc.tl.rank_genes_groups(adataKnn, groupby="leiden", mehthod="wilcoxon", pts=True, layer="logCPU")

## Unfiltered scanpy results

In [ ]:
sc.pl.rank_genes_groups_matrixplot(adataKnn, n_genes=10,min_logfoldchange=2, values_to_plot="logfoldchanges", vmin=-3, vmax=3, cmap="coolwarm")

## Filtered scanpy results

In [ ]:


maxFDR = 0.05
min_pct = 0.1
minlogFC = 1

scanpyDEGs = pd.DataFrame()
for group in adataKnn.obs["leiden"].unique():
    localDEGs = sc.get.rank_genes_groups_df(adataKnn, group)
    localDEGs = localDEGs[localDEGs["pct_nz_group"] > min_pct].copy()
    localDEGs = localDEGs[localDEGs["logfoldchanges"] > minlogFC].copy()
    localDEGs = localDEGs[localDEGs["pvals_adj"] < maxFDR].copy()
    localDEGs["leiden"] = group
    rank = (
        -np.log10(np.clip(localDEGs["pvals_adj"], localDEGs.loc[localDEGs["pvals_adj"] != 0,"pvals_adj"].min(), None))
        * np.sign(localDEGs["logfoldchanges"]) * localDEGs["pct_nz_group"]
    )
    localDEGs["rank"] = rank
    scanpyDEGs = pd.concat([scanpyDEGs,localDEGs ])


saveDir = f"DEGS_{nb_name}_scanpy"
os.makedirs(saveDir, exist_ok=True)

for leiden in scanpyDEGs["leiden"].unique():
    localDegs = scanpyDEGs[scanpyDEGs["leiden"] == leiden].copy()  
    localDegs.to_excel(os.path.join(saveDir, f"DEGS_{leiden}_maxFDR{maxFDR}.xlsx"), index=False)



out = run_marker_selection_and_ordering(
    adata=adataKnn,
    degs_df=scanpyDEGs,
    deg_group_col="leiden",   # who owns candidate marker genes
    obs_groupby="leiden",  # where expression is summarized
    gene_col="names",
    top_n=10,
    fdr_col="pvals_adj",layer=None, 
    logfc_col="logfoldchanges",sort_by="rank", ascending=False,
    min_logfc=0.5,
    fdr_thresh=0.01,
)



sc.pl.dotplot(
    adataKnn,
    out["ordered_genes"],
    "leiden",
    categories_order=out["ordered_groups"],
    layer=None,
    cmap="RdBu_r",standard_scale="var",
)

sc.pl.matrixplot(
    adataKnn,
    out["ordered_genes"],
    "leiden",
    categories_order=out["ordered_groups"],
    layer=None,
    cmap="RdBu_r",standard_scale="var",
)


# 2) Now Seurat method

In [ ]:
from transferUtils import *

export_anndata_minimal(
    adata=adataKnn,
    out_dir=f"./SeuratReady_knn_metacells{Celltype}_{DSname}",
    base=f"{Celltype}_{DSname}",
    layer="counts",                 # ignored for data, used only for shape checking
    obsm_key="X_umap",
    replace_counts_with_zeros=False, # <- zeros
)

In [ ]:
%%R -i DSname -i Celltype  -i homeDir  -o CtMarkers -i maxFDR -i min_pct
library(Seurat)
library(dplyr)

source(paste0(homeDir,"/utils/transferUtils.R"))


# If needed, pin Python with SciPy before calling:
# reticulate::use_python("/usr/bin/python", required = TRUE)

sce <- load_export_as_sce(
  out_dir = sprintf("SeuratReady_knn_metacells%s_%s", Celltype, DSname),
  base = sprintf("%s_%s", Celltype, DSname),
  reduced_name = "X_umap",
  add_coords_to_reduced = TRUE,
  include_coords_in_coldata = TRUE,
  counts_transpose = TRUE,
  sep = "\t"
)

SeuratObject <- as.Seurat(sce, counts = "counts", data = "counts")
SeuratObject <- NormalizeData(SeuratObject)
Idents(SeuratObject) <- "leiden"
# markers <- FindAllMarkers(SeuratObject, only.pos = TRUE)

CtMarkers<-FindAllMarkers(SeuratObject,only.pos = T,min.pct = min_pct)
CtMarkers<-CtMarkers[CtMarkers$p_val_adj<maxFDR,]

In [ ]:
CtMarkers

In [ ]:
saveDir = f"DEGS_{nb_name}_Seurat"
os.makedirs(saveDir, exist_ok=True)


rank = (
    -np.log10(np.clip(CtMarkers["p_val_adj"], CtMarkers.loc[CtMarkers["p_val_adj"] != 0,"p_val_adj"].min(), None))
    * np.sign(CtMarkers["avg_log2FC"])
    * CtMarkers["pct.1"]
)
CtMarkers["rank"] = rank

for AnnotatedDomain in CtMarkers["cluster"].unique():
    localDegs = CtMarkers[CtMarkers["cluster"] == AnnotatedDomain].copy()  
    localDegs.to_excel(os.path.join(saveDir, f"DEGS_{AnnotatedDomain}_maxFDR{maxFDR}_minPosRate{min_pct}.xlsx"), index=False)

# Also plot Seurat filtered results

In [ ]:
MarkersDict ={}
topN = 10
for i in CtMarkers["cluster"].unique():
    MarkersDict[i] = CtMarkers[CtMarkers["cluster"] == i].sort_values("avg_log2FC").tail(topN)["gene"].tolist()

adataKnn.X = adataKnn.layers["counts"].copy()

## Filtered logFC order

In [ ]:
sc.pl.matrixplot(
    adataKnn,
    MarkersDict,
    "leiden",
    dendrogram=True,
    # colorbar_title="mean z-score",
    # layer="scaled",
    cmap="RdBu_r",standard_scale="var",
)


sc.pl.dotplot(
    adataKnn,
    MarkersDict,
    "leiden",
    dendrogram=True,
    # colorbar_title="mean z-score",
    # layer="scaled",
    cmap="RdBu_r",standard_scale="var",
)


## Filtered  reordered by group

In [ ]:


out = run_marker_selection_and_ordering(
    adata=adataKnn,
    degs_df=CtMarkers,
    deg_group_col="cluster",   # who owns candidate marker genes
    obs_groupby="leiden",  # where expression is summarized
    gene_col="gene",
    top_n=10,
    fdr_col="p_val_adj",
    logfc_col="avg_log2FC",layer="logCPU", sort_by="rank", ascending=False,
    min_logfc=0.5,
    fdr_thresh=0.01,
)


sc.pl.matrixplot(
    adataKnn,
    out["ordered_genes"],
    "leiden",
    categories_order=out["ordered_groups"],
    # colorbar_title="mean z-score",
    layer="logCPU",
    cmap="RdBu_r",standard_scale="var",
)

# Signature scoring

In [ ]:
out_dir_export = f"{nb_name}_AUCready"
out_dir_export

In [ ]:
from transferUtils import *


export_anndata_minimal(
    adata=adataKnn,
    out_dir=f"./{out_dir_export}",
    base=f"adataKnn",
    layer="counts",                 # ignored for data, used only for shape checking
    obsm_key="X_pca",
    replace_counts_with_zeros=False, # <- zeros
)

## 1 Bulk-derived

In [ ]:
import os
import glob
import pandas as pd
import rapids_singlecell as rsc

topN = 300

BaseDir = "/data/projects/spatialTX/0_SignaturePrep/Contrasts_RAW_signatures_MB_MELA_NCRE_Neurons/"
signature_list = []

for filepath in glob.glob(os.path.join(BaseDir, "*.xlsx")):
    sig = pd.read_excel(filepath)

    # safer: split only the filename, not the full path
    fname = os.path.basename(filepath)
    sig_name = fname.split("_")[1]   # adjust index if needed after checking filenames

    # keep top N rows
    sig = sig[(sig["FDR"] < 0.01) & (sig["Celltypes_upregulated"] == sig_name) & (sig["genes"].isin(adataKnn.var_names))].sort_values("logFC", ascending=False).head(topN)
    print(f"{len(sig)} Genes met requirements for {sig_name}")
    signature_list.append(sig)

signatureDF_verrillo = pd.concat(signature_list, ignore_index=True)

In [ ]:
import numpy as np

# remove old score columns
old_cols = [col for col in adataKnn.obs.columns if "scoregenes_verrillo" in col]
adataKnn.obs.drop(columns=old_cols, inplace=True, errors="ignore")

# restore expression matrix
adataKnn.X = adataKnn.layers["logCPU"].copy()

# compute scores
rsc.get.anndata_to_GPU(adataKnn)
rsc.pp.scale(adataKnn, zero_center=False)

for celltype in signatureDF_verrillo["celltype"].dropna().unique():
    genes = signatureDF_verrillo.loc[signatureDF_verrillo["celltype"] == celltype, "genes"].tolist()
    genes = [g for g in genes if g in adataKnn.var_names]  # recommended
    
    if len(genes) == 0:
        print(f"Skipping {celltype}: no genes found in adata.var_names")
        continue

    sigName = f"scoregenes_verrillo{celltype}"
    rsc.tl.score_genes(adataKnn, gene_list=genes, score_name=sigName)

rsc.get.anndata_to_CPU(adataKnn)
adataKnn.X = adataKnn.layers["logCPU"].copy()

# collect NEW score columns after scoring
cols = [col for col in adataKnn.obs.columns if "scoregenes_verrillo" in col]

if len(cols) == 0:
    raise ValueError("No scoregenes_verrillo columns were created.")

q99List = [adataKnn.obs[col].quantile(0.99) for col in cols]
q1list  = [adataKnn.obs[col].quantile(0.01) for col in cols]

vmax = np.max(q99List)
vmin = np.min(q1list)

sc.pl.umap(
    adataKnn,
    color=cols,
    size=50,
    cmap="Greens",
    vmax=vmax,
    vmin=vmin
)

In [ ]:
for scorecol in cols:
   plot_spatial_obs(
    adataKnn, obs=scorecol, width=7, dpi=200,dotscale=4 ,img_key="hires",marker='o',linewidths=0,vmin=vmin, vmax=vmax,palette="Greens",
    spatial_key='spatial',scale_key="tissue_hires_scalef",library_id= f"{FigTag}_hires_image", img_alpha=.4,  legend_kwargs={"fontsize":10}) 

In [ ]:
%%R  -i homeDir -i out_dir_export


library(igraph)
library(ggraph)

library(ggplot2)
library(dplyr)

source(paste0(homeDir,"/utils/transferUtils.R"))

sce <- load_export_as_sce(
  out_dir = paste0("./", out_dir_export),
  base = "adataKnn",
  reduced_name = "X_pca",
  add_coords_to_reduced = TRUE,
  include_coords_in_coldata = TRUE,
  counts_transpose = TRUE,
  sep = "\t"
)


exprMatrix  <- assay(sce, "counts")
dim(exprMatrix )

## AUCELL

In [ ]:
%%R  -i homeDir -i signatureDF_verrillo -w 1500 -h 1500 -o df_auc0

library(GSEABase)
library(AUCell)
as_gene_vector <- function(x) {
  if (is.null(x)) return(character(0))
  x <- as.character(x)
  x <- trimws(x)
  x <- x[!is.na(x) & nzchar(x)]
  unique(x)
}

# split long table -> list of character vectors
Signatures <- split(signatureDF_verrillo$gene, signatureDF_verrillo$celltype)
Signatures <- lapply(Signatures, as_gene_vector)

# drop empty sets
Signatures <- Signatures[lengths(Signatures) > 0]

# build GeneSetCollection
geneSets <- GeneSetCollection(
  mapply(
    function(ids, nm) GeneSet(geneIds = ids, setName = nm),
    ids = Signatures,
    nm  = names(Signatures),
    SIMPLIFY = FALSE
  )
)

# quick check
length(geneSets)
names(geneSets)[1:min(10, length(geneSets))]


setClassUnion("ExpData", c("matrix", "SummarizedExperiment"))

# Run aucell
cells_rankings <- AUCell_buildRankings(exprMatrix, plotStats=TRUE)
cells_AUC <- AUCell_calcAUC(geneSets, cells_rankings)


set.seed(333)
par(mfrow=c(5,5)) 
cells_assignment <- AUCell_exploreThresholds(cells_AUC, plotHist=TRUE, assign=TRUE)

selectedThresholds <- getThresholdSelected(cells_assignment)
auc_mat <- getAUC(cells_AUC)   # rows = signatures, cols = barcodes

# keep only signatures present in auc_mat
selectedThresholds <- selectedThresholds[intersect(names(selectedThresholds), rownames(auc_mat))]

masked_list <- list()

for (geneSetName in names(selectedThresholds)) {
  auc_vec <- auc_mat[geneSetName, ]
  thr <- selectedThresholds[[geneSetName]]

  #auc_vec[auc_vec <= thr] <- 0  # Option B: AUC if pass else 0
  masked_list[[paste0("AUC_score_verrillo_", geneSetName)]] <- auc_vec
}

# barcode as rownames (index), not a column
df_auc0 <- as.data.frame(masked_list, check.names = FALSE)

In [ ]:
for col in adataKnn.obs.columns:
    if col.startswith("AUC_score_verrillo_"):
        del adataKnn.obs[col]

for i in df_auc0.columns:
    if i in adataKnn.obs.columns:
        del adataKnn.obs[i]

adataKnn.obs = pd.concat([adataKnn.obs, df_auc0], axis = 1)
cols = [c for c in adataKnn.obs.columns if c.startswith("AUC_score_verrillo_")]


q99List = []
q1list = []
for scorecol in cols:
    print(scorecol)
    q99List.append(adataKnn.obs[scorecol].quantile(0.99))
    q1list.append(adataKnn.obs[scorecol].quantile(0.01))
vmax = np.max(q99List)  
vmin = np.max(q1list)  



sc.pl.umap(adataKnn, color=cols, size=50,cmap="Greens", 
           vmax=vmax, 
          vmin=vmin)


for scorecol in cols:
   plot_spatial_obs(
    adataKnn, obs=scorecol, width=7, dpi=200,dotscale=4 ,img_key="hires",marker='o',linewidths=0,vmin=vmin, vmax=vmax,palette="Greens",
    spatial_key='spatial',scale_key="tissue_hires_scalef",library_id= f"{FigTag}_hires_image", img_alpha=.4,  legend_kwargs={"fontsize":10}) 

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# optional: nicer default sizing/fonts without seaborn
plt.rcParams["figure.dpi"] = 140
plt.rcParams["axes.spines.top"] = False
plt.rcParams["axes.spines.right"] = False
plt.rcParams["axes.titleweight"] = "bold"

celltypes = list(signatureDF_verrillo["celltype"].dropna().unique())
n = len(celltypes)

# panel layout
ncols = min(3, n)
nrows = int(np.ceil(n / ncols))

fig, axes = plt.subplots(nrows, ncols, figsize=(5.5 * ncols, 4.8 * nrows))
axes = np.atleast_1d(axes).ravel()

for ax, celltype in zip(axes, celltypes):
    xcol = f"scoregenes_verrillo{celltype}"
    ycol = f"AUC_score_verrillo_{celltype}"
    
    if xcol not in adataKnn.obs.columns or ycol not in adataKnn.obs.columns:
        ax.set_visible(False)
        print(f"xcol not found {xcol}")
        continue

    df = adataKnn.obs[[xcol, ycol]].copy()
    df = df.replace([np.inf, -np.inf], np.nan).dropna()

    if df.empty:
        ax.set_visible(False)
        continue

    x = df[xcol].to_numpy()
    y = df[ycol].to_numpy()

    # scatter
    ax.scatter(
        x, y,
        s=16,
        alpha=0.45,
        edgecolors="none"
    )

    # regression line
    if len(df) >= 2 and np.std(x) > 0 and np.std(y) > 0:
        m, b = np.polyfit(x, y, 1)
        xx = np.linspace(x.min(), x.max(), 200)
        yy = m * xx + b
        ax.plot(xx, yy, linewidth=2.2)

        r = np.corrcoef(x, y)[0, 1]
        ax.text(
            0.03, 0.97,
            f"r = {r:.2f}\nn = {len(df)}",
            transform=ax.transAxes,
            ha="left", va="top",
            bbox=dict(boxstyle="round,pad=0.3", facecolor="white", alpha=0.8, edgecolor="0.8")
        )

    ax.set_title(celltype)
    ax.set_xlabel("ScoreGenes")
    ax.set_ylabel("AUCell")
    ax.grid(True, alpha=0.2)

# hide unused panels
for ax in axes[len(celltypes):]:
    ax.set_visible(False)

fig.suptitle("ScoreGenes vs AUCell by cell type", y=1.02, fontsize=16, fontweight="bold")
fig.tight_layout()
plt.show()

## 2 pozniak signature

In [ ]:
topN = 100
CS = pd.read_csv("/data/projects/spatialTX/data/Signatures/Pozniak/Pozniak_signature.csv")
CS = CS[CS["p_val_adj"] < 0.01].copy()
CS = CS[CS["avg_logFC"] >= .2].copy()
CS = CS[CS["gene"].isin(adataKnn.var_names)].copy()


# If celltype is blank on many rows (common in Excel-like exports)
CS["cluster"] = CS["cluster"].ffill()

# Drop any remaining missing essentials
CS = CS.dropna(subset=["cluster", "gene", "p_val_adj","avg_logFC"])

# Ensure strings
CS["cluster"] = CS["cluster"].astype(str)
CS["gene"] = CS["gene"].astype(str)

# TopN per celltype
###################################################################### Sorted by INCREASING FDR
signatureDF_pozniak = (CS.sort_values(["cluster", "avg_logFC"], ascending=[True, False])
            .groupby("cluster", sort=False)
            .head(topN)[["cluster", "gene"]]
            .rename(columns={"gene": "gene"}))


print("NA cluster:", signatureDF_pozniak["cluster"].isna().sum())
print("Unique cluster:", signatureDF_pozniak["cluster"].nunique())
print(signatureDF_pozniak["cluster"].value_counts())

signatureDF_pozniak = signatureDF_pozniak.rename(columns={"cluster":"celltype","gene":"genes"})


In [ ]:
import numpy as np

# remove old score columns
old_cols = [col for col in adataKnn.obs.columns if "scoregenes_pozniak" in col]
adataKnn.obs.drop(columns=old_cols, inplace=True, errors="ignore")

# restore expression matrix
adataKnn.X = adataKnn.layers["logCPU"].copy()

# compute scores
rsc.get.anndata_to_GPU(adataKnn)
rsc.pp.scale(adataKnn, zero_center=False)

for celltype in signatureDF_pozniak["celltype"].dropna().unique():
    genes = signatureDF_pozniak.loc[signatureDF_pozniak["celltype"] == celltype, "genes"].tolist()
    genes = [g for g in genes if g in adataKnn.var_names]  # recommended
    
    if len(genes) == 0:
        print(f"Skipping {celltype}: no genes found in adata.var_names")
        continue

    sigName = f"scoregenes_pozniak{celltype}"
    rsc.tl.score_genes(adataKnn, gene_list=genes, score_name=sigName)

rsc.get.anndata_to_CPU(adataKnn)
adataKnn.X = adataKnn.layers["logCPU"].copy()

# collect NEW score columns after scoring
cols = [col for col in adataKnn.obs.columns if "scoregenes_pozniak" in col]

if len(cols) == 0:
    raise ValueError("No scoregenes_pozniak columns were created.")

q99List = [adataKnn.obs[col].quantile(0.99) for col in cols]
q1list  = [adataKnn.obs[col].quantile(0.01) for col in cols]

vmax = np.max(q99List)
vmin = np.min(q1list)

sc.pl.umap(
    adataKnn,
    color=cols,
    size=50,
    cmap="Greens",
    vmax=vmax,
    vmin=vmin
)

for scorecol in cols:
   plot_spatial_obs(
    adataKnn, obs=scorecol, width=7, dpi=200,dotscale=4 ,img_key="hires",marker='o',linewidths=0,vmin=vmin, vmax=vmax,palette="Greens",
    spatial_key='spatial',scale_key="tissue_hires_scalef",library_id= f"{FigTag}_hires_image", img_alpha=.4,  legend_kwargs={"fontsize":10}) 




## AUCELL

In [ ]:
%%R  -i homeDir -i out_dir_export


library(igraph)
library(ggraph)

library(ggplot2)
library(dplyr)

source(paste0(homeDir,"/utils/transferUtils.R"))

sce <- load_export_as_sce(
  out_dir = paste0("./", out_dir_export),
  base = "adataKnn",
  reduced_name = "X_pca",
  add_coords_to_reduced = TRUE,
  include_coords_in_coldata = TRUE,
  counts_transpose = TRUE,
  sep = "\t"
)


exprMatrix  <- assay(sce, "counts")
dim(exprMatrix )

In [ ]:
%%R  -i homeDir -i signatureDF_pozniak -w 1500 -h 1500 -o df_auc0

library(GSEABase)
library(AUCell)
as_gene_vector <- function(x) {
  if (is.null(x)) return(character(0))
  x <- as.character(x)
  x <- trimws(x)
  x <- x[!is.na(x) & nzchar(x)]
  unique(x)
}

# split long table -> list of character vectors
Signatures <- split(signatureDF_pozniak$gene, signatureDF_pozniak$celltype)
Signatures <- lapply(Signatures, as_gene_vector)

# drop empty sets
Signatures <- Signatures[lengths(Signatures) > 0]

# build GeneSetCollection
geneSets <- GeneSetCollection(
  mapply(
    function(ids, nm) GeneSet(geneIds = ids, setName = nm),
    ids = Signatures,
    nm  = names(Signatures),
    SIMPLIFY = FALSE
  )
)

# quick check
length(geneSets)
names(geneSets)[1:min(10, length(geneSets))]


setClassUnion("ExpData", c("matrix", "SummarizedExperiment"))

# Run aucell
cells_rankings <- AUCell_buildRankings(exprMatrix, plotStats=TRUE)
cells_AUC <- AUCell_calcAUC(geneSets, cells_rankings)


set.seed(333)
par(mfrow=c(5,5)) 
cells_assignment <- AUCell_exploreThresholds(cells_AUC, plotHist=TRUE, assign=TRUE)

selectedThresholds <- getThresholdSelected(cells_assignment)
auc_mat <- getAUC(cells_AUC)   # rows = signatures, cols = barcodes

# keep only signatures present in auc_mat
selectedThresholds <- selectedThresholds[intersect(names(selectedThresholds), rownames(auc_mat))]

masked_list <- list()

for (geneSetName in names(selectedThresholds)) {
  auc_vec <- auc_mat[geneSetName, ]
  thr <- selectedThresholds[[geneSetName]]

  #auc_vec[auc_vec <= thr] <- 0  # Option B: AUC if pass else 0
  masked_list[[paste0("AUC_score_pozniak_", geneSetName)]] <- auc_vec
}

# barcode as rownames (index), not a column
df_auc0 <- as.data.frame(masked_list, check.names = FALSE)

In [ ]:
for col in adataKnn.obs.columns:
    if col.startswith("AUC_score_pozniak_"):
        del adataKnn.obs[col]

for i in df_auc0.columns:
    if i in adataKnn.obs.columns:
        del adataKnn.obs[i]

adataKnn.obs = pd.concat([adataKnn.obs, df_auc0], axis = 1)
cols = [c for c in adataKnn.obs.columns if c.startswith("AUC_score_pozniak_")]


q99List = []
q1list = []
for scorecol in cols:
    print(scorecol)
    q99List.append(adataKnn.obs[scorecol].quantile(0.99))
    q1list.append(adataKnn.obs[scorecol].quantile(0.01))
vmax = np.max(q99List)  
vmin = np.max(q1list)  



sc.pl.umap(adataKnn, color=cols, size=50,cmap="Greens", 
           vmax=vmax, 
          vmin=vmin)


for scorecol in cols:
   plot_spatial_obs(
    adataKnn, obs=scorecol, width=7, dpi=200,dotscale=4 ,img_key="hires",marker='o',linewidths=0,vmin=vmin, vmax=vmax,palette="Greens",
    spatial_key='spatial',scale_key="tissue_hires_scalef",library_id= f"{FigTag}_hires_image", img_alpha=.4,  legend_kwargs={"fontsize":10}) 

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# optional: nicer default sizing/fonts without seaborn
plt.rcParams["figure.dpi"] = 140
plt.rcParams["axes.spines.top"] = False
plt.rcParams["axes.spines.right"] = False
plt.rcParams["axes.titleweight"] = "bold"

celltypes = list(signatureDF_pozniak["celltype"].dropna().unique())
n = len(celltypes)

# panel layout
ncols = min(3, n)
nrows = int(np.ceil(n / ncols))

fig, axes = plt.subplots(nrows, ncols, figsize=(5.5 * ncols, 4.8 * nrows))
axes = np.atleast_1d(axes).ravel()

for ax, celltype in zip(axes, celltypes):
    xcol = f"scoregenes_pozniak{celltype}"
    ycol = f"AUC_score_pozniak_{celltype}"
    
    if xcol not in adataKnn.obs.columns or ycol not in adataKnn.obs.columns:
        ax.set_visible(False)
        print(f"xcol not found {xcol}")
        continue

    df = adataKnn.obs[[xcol, ycol]].copy()
    df = df.replace([np.inf, -np.inf], np.nan).dropna()

    if df.empty:
        ax.set_visible(False)
        continue

    x = df[xcol].to_numpy()
    y = df[ycol].to_numpy()

    # scatter
    ax.scatter(
        x, y,
        s=16,
        alpha=0.45,
        edgecolors="none"
    )

    # regression line
    if len(df) >= 2 and np.std(x) > 0 and np.std(y) > 0:
        m, b = np.polyfit(x, y, 1)
        xx = np.linspace(x.min(), x.max(), 200)
        yy = m * xx + b
        ax.plot(xx, yy, linewidth=2.2)

        r = np.corrcoef(x, y)[0, 1]
        ax.text(
            0.03, 0.97,
            f"r = {r:.2f}\nn = {len(df)}",
            transform=ax.transAxes,
            ha="left", va="top",
            bbox=dict(boxstyle="round,pad=0.3", facecolor="white", alpha=0.8, edgecolor="0.8")
        )

    ax.set_title(celltype)
    ax.set_xlabel("ScoreGenes")
    ax.set_ylabel("AUCell")
    ax.grid(True, alpha=0.2)

# hide unused panels
for ax in axes[len(celltypes):]:
    ax.set_visible(False)

fig.suptitle("ScoreGenes vs AUCell by cell type", y=1.02, fontsize=16, fontweight="bold")
fig.tight_layout()
plt.show()